#### 1. CONFIGURACIÓN E IMPORTACIONES

In [3]:
print("--- 1. Importando librerías ---")
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import joblib  # Para guardar los modelos

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Descargar recursos de NLTK (solo necesario la primera vez que se ejecuta)
try:
    nltk.data.find('corpora/stopwords')
    nltk.data.find('corpora/wordnet')
except LookupError:
    print("Descargando recursos de NLTK (stopwords, wordnet)...")
    nltk.download('stopwords')
    nltk.download('wordnet')

print("Librerías importadas y configuración completa.\n")

--- 1. Importando librerías ---
Descargando recursos de NLTK (stopwords, wordnet)...
Librerías importadas y configuración completa.



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Omar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Omar\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


#### 2. CARGA DE DATOS Y PREPROCESAMIENTO

In [4]:
print("--- 2. Cargando datos y aplicando preprocesamiento ---")

file_path = '../data/youtube_comment_dataset.csv'
df = pd.read_csv(file_path)
df.dropna(subset=['Text'], inplace=True)

# 2.1 Definir la función de preprocesamiento de texto
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Eliminar URLs
    text = re.sub(r'<.*?>+', '', text) # Eliminar HTML
    text = re.sub(r'[^a-z\s]', '', text) # Solo letras y espacios
    tokens = text.split()
    clean_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(clean_tokens)

# 2.2 Aplicar el preprocesamiento al DataFrame
print("Aplicando preprocesamiento de texto a la columna 'Text'...")
df['Processed_Text'] = df['Text'].apply(preprocess_text)
print("Preprocesamiento completado.")

# 2.3 Crear la variable objetivo 'IsHate'
label_cols = [col for col in df.columns if col.startswith('Is')]
for col in label_cols:
    df[col] = df[col].apply(lambda x: 1 if str(x).upper() == 'TRUE' else 0)
df['IsHate'] = df[label_cols].any(axis=1).astype(int)

# Verificación de que el preprocesamiento funcionó
print("\nEjemplo de preprocesamiento:")
print("  Texto Original:", df['Text'].iloc[2])
print("  Texto Procesado:", df['Processed_Text'].iloc[2], "\n")

--- 2. Cargando datos y aplicando preprocesamiento ---
Aplicando preprocesamiento de texto a la columna 'Text'...
Preprocesamiento completado.

Ejemplo de preprocesamiento:
  Texto Original: 
Dont you reckon them 'black lives matter' banners being held by white cunts is  kinda patronizing and ironically racist. could they have not come up with somethin better.. or is it just what white folks do to give them selves pride. 'ooo look at me im being nice for the black people' why does it always have to be about race actually the whole world is pussyfootin around for fear of being racist. its fuckin daft man.
  Texto Procesado: dont reckon black life matter banner held white cunt kinda patronizing ironically racist could come somethin better white folk give self pride ooo look im nice black people always race actually whole world pussyfootin around fear racist fuckin daft man 



#### 3. DIVISIÓN DE DATOS EN ENTRENAMIENTO Y PRUEBA

In [5]:
print("--- 3. Dividiendo los datos en conjuntos de entrenamiento y prueba ---")

X = df['Processed_Text']
y = df['IsHate']

# Dividimos 80% para entrenar, 20% para probar
# random_state=42 asegura que la división sea siempre la misma (reproducibilidad)
# stratify=y asegura que la proporción de clases 0 y 1 sea la misma en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Tamaño del conjunto de entrenamiento: {len(X_train)} muestras")
print(f"Tamaño del conjunto de prueba: {len(X_test)} muestras\n")

--- 3. Dividiendo los datos en conjuntos de entrenamiento y prueba ---
Tamaño del conjunto de entrenamiento: 800 muestras
Tamaño del conjunto de prueba: 200 muestras



#### MODELO 1 - BASELINE (REGRESIÓN LOGÍSTICA)

In [6]:
print("--- 4. Entrenando y evaluando el modelo base: Regresión Logística ---")

# 4.1 Definir la Pipeline del modelo
# Una Pipeline encadena pasos: aquí, vectorizar el texto y luego aplicar el clasificador.
pipeline_logreg = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000))
])

# 4.2 Entrenar el modelo
pipeline_logreg.fit(X_train, y_train)

# 4.3 Realizar predicciones y evaluar
y_pred_logreg = pipeline_logreg.predict(X_test)

print("\nReporte de Clasificación (Regresión Logística en Test):")
print(classification_report(y_test, y_pred_logreg, target_names=['No Odio', 'Odio']))

--- 4. Entrenando y evaluando el modelo base: Regresión Logística ---

Reporte de Clasificación (Regresión Logística en Test):
              precision    recall  f1-score   support

     No Odio       0.73      0.76      0.74       108
        Odio       0.70      0.66      0.68        92

    accuracy                           0.71       200
   macro avg       0.71      0.71      0.71       200
weighted avg       0.71      0.71      0.71       200



#### MODELO 2 - ENSEMBLE (XGBOOST)

In [7]:
print("\n--- 5. Entrenando y evaluando el modelo de ensemble: XGBoost ---")

# 5.1 Definir la Pipeline del modelo
pipeline_xgb = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
])

# 5.2 Entrenar el modelo
pipeline_xgb.fit(X_train, y_train)

# 5.3 Realizar predicciones y evaluar
y_pred_xgb = pipeline_xgb.predict(X_test)

print("\nReporte de Clasificación (XGBoost en Test):")
print(classification_report(y_test, y_pred_xgb, target_names=['No Odio', 'Odio']))


--- 5. Entrenando y evaluando el modelo de ensemble: XGBoost ---

Reporte de Clasificación (XGBoost en Test):
              precision    recall  f1-score   support

     No Odio       0.70      0.80      0.75       108
        Odio       0.72      0.61      0.66        92

    accuracy                           0.71       200
   macro avg       0.71      0.70      0.70       200
weighted avg       0.71      0.71      0.71       200



c:\Users\Omar\Desktop\trabajo\repositorios\p-x-nlp-feel-recognize\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:47:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


#### 6. ANÁLISIS DE RESULTADOS Y CONTROL DE OVERFITTING

In [8]:
print("\n--- 6. Análisis de resultados y control de overfitting ---")

print("Comparación del F1-Score para la clase 'Odio' en el conjunto de prueba:")
f1_logreg = f1_score(y_test, y_pred_logreg)
f1_xgb = f1_score(y_test, y_pred_xgb)
print(f"  - Regresión Logística: {f1_logreg:.4f}")
print(f"  - XGBoost:             {f1_xgb:.4f}")

# Comprobación de overfitting en el modelo más complejo (XGBoost)
print("\nComprobando overfitting para XGBoost...")
y_pred_train_xgb = pipeline_xgb.predict(X_train)

f1_train = f1_score(y_train, y_pred_train_xgb)
f1_test = f1_xgb
overfitting_diff = (f1_train - f1_test) * 100

print(f"  - F1-Score en TRAIN: {f1_train:.4f}")
print(f"  - F1-Score en TEST:  {f1_test:.4f}")
print(f"  - Diferencia: {overfitting_diff:.2f}%")

if overfitting_diff > 5:
    print("\n  ALERTA: Se ha detectado un posible overfitting (>5% de diferencia).")
    print("  Esto se abordará en la fase de optimización de hiperparámetros (Optuna).\n")
else:
    print("\n  El modelo muestra un buen grado de generalización.\n")


--- 6. Análisis de resultados y control de overfitting ---
Comparación del F1-Score para la clase 'Odio' en el conjunto de prueba:
  - Regresión Logística: 0.6816
  - XGBoost:             0.6588

Comprobando overfitting para XGBoost...
  - F1-Score en TRAIN: 0.9437
  - F1-Score en TEST:  0.6588
  - Diferencia: 28.48%

  ALERTA: Se ha detectado un posible overfitting (>5% de diferencia).
  Esto se abordará en la fase de optimización de hiperparámetros (Optuna).



In [9]:
print("--- 7. Conclusiones y próximos pasos ---")

if f1_xgb > f1_logreg:
    print("El modelo XGBoost ha obtenido un mejor F1-Score y es el 'modelo campeón' inicial.")
    best_pipeline = pipeline_xgb
    model_name = "xgb_pipeline.joblib"
else:
    print("El modelo de Regresión Logística ha obtenido un mejor F1-Score y es el 'modelo campeón' inicial.")
    best_pipeline = pipeline_logreg
    model_name = "logreg_pipeline.joblib"

# Guardar el mejor modelo para usarlo en la API más adelante
joblib.dump(best_pipeline, model_name)
print(f"\nEl mejor modelo ha sido guardado como '{model_name}' en la carpeta actual.")

conclusions = """
### Resumen del Entrenamiento:
Se han entrenado dos modelos para la clasificación binaria de comentarios de odio.
- **Baseline (Regresión Logística):** Ofrece un rendimiento sólido y es computacionalmente eficiente.
- **Ensemble (XGBoost):** Generalmente logra un mejor rendimiento predictivo, aunque es más propenso al overfitting inicial.

El modelo con el F1-Score más alto para la clase 'Odio' ha sido seleccionado como nuestro mejor modelo base.

### Próximos Pasos en el Proyecto:
1.  **Optimización de Hiperparámetros:** Utilizar Optuna (o una herramienta similar) para ajustar los hiperparámetros del modelo campeón y mejorar su rendimiento, especialmente para controlar el overfitting.
2.  **Modelos Avanzados:** Implementar y comparar estos resultados con una Red Neuronal (LSTM) y un modelo basado en Transformers (BERT).
3.  **Backend:** Una vez tengamos el modelo final, lo integraremos en una API con FastAPI para servir las predicciones.
4.  **Tests Unitarios:** Añadir tests para validar la función de preprocesamiento y la API.
"""

print(conclusions)

--- 7. Conclusiones y próximos pasos ---
El modelo de Regresión Logística ha obtenido un mejor F1-Score y es el 'modelo campeón' inicial.

El mejor modelo ha sido guardado como 'logreg_pipeline.joblib' en la carpeta actual.

### Resumen del Entrenamiento:
Se han entrenado dos modelos para la clasificación binaria de comentarios de odio.
- **Baseline (Regresión Logística):** Ofrece un rendimiento sólido y es computacionalmente eficiente.
- **Ensemble (XGBoost):** Generalmente logra un mejor rendimiento predictivo, aunque es más propenso al overfitting inicial.

El modelo con el F1-Score más alto para la clase 'Odio' ha sido seleccionado como nuestro mejor modelo base.

### Próximos Pasos en el Proyecto:
1.  **Optimización de Hiperparámetros:** Utilizar Optuna (o una herramienta similar) para ajustar los hiperparámetros del modelo campeón y mejorar su rendimiento, especialmente para controlar el overfitting.
2.  **Modelos Avanzados:** Implementar y comparar estos resultados con una Red N